# Simplified Chinese support and reach on Steam

**DATA2010 Data Science Programming · final project**

The notebook follows the same sections as the README.

| Section | What it does | Output |
| :--- | :--- | :--- |
| I. Setup and how to reproduce | how to get the project and run it | |
| II.1 Business understanding | the question | |
| II.2 Data collection | read the two raw files | |
| II.3 Data processing | 3.1 clean · 3.2 SQ1 groups · 3.3 SQ2 groups · 3.4 save the tables | `pcd_games_clean`, `pcd_sq1_groups`, `pcd_sq2_groups` |
| II.4 Data analysis / EDA | describe the three tables | `report/figures/` |
| II.5 Data modelling | compare the groups, answer SQ1-SQ3 | `report/figures/` |
| II.6 Model deployment | | |

**How to read it**
- Each code cell = one step, labelled 2.1, 2.2, ..., 3.1.1, 3.1.2, ...; in section 3.1, each problem of a step has a letter (3.1.1a, 3.1.1b, ...) used in the table and in the code comments.
- Its first lines say: `Reason` (why), `Rule` (what is done), `Not used ... because` (what was left out).
- `DATA ISSUE` = a problem found in the data.

# I. Setup and how to reproduce

**Needed:** [Git](https://git-scm.com) with [Git LFS](https://git-lfs.com) (the Windows installer of Git already includes it), Python 3.12 or 3.13.

**Steps**
1. `git lfs install` (once per computer)
2. `git clone https://github.com/theonlypgk/data2010-steam-chinese.git`
3. `cd data2010-steam-chinese` then `pip install -r requirements.txt`
4. Open `scripts/steam_chinese_analysis.ipynb` and choose *Run All*.

- `data/raw/games_a.csv` is only a few lines of text (not ~400 MB)? → run `git lfs pull` in the project folder.

In [ ]:
# I.a · Import the libraries and name the folders
# Reason: three libraries are enough; paths are relative to this notebook (scripts/)

import re                     # text patterns (clean the list cells, find Chinese characters)

import numpy as np
import pandas as pd

RAW = "../data/raw/"                          # the two input files
PROCESSED_CSV = "../data/processed/csv/"      # output tables
FIGURES = "../report/figures/"                # saved figures
PREFIX = "pcd_"                               # file prefix of every output table
print("pandas", pd.__version__, "| numpy", np.__version__)

# II. Data science life cycle

## 1.​ Business understanding

**Main question:** should a Steam game support Simplified Chinese, for which genres, and when?

| | Question |
|---|---|
| SQ1 | How much larger is the reach of games with Simplified Chinese, and in which genres? |
| SQ2 | Does adding Simplified Chinese after launch still increase reach, and in which genres? |
| SQ3 | For each genre: translate at launch, later, or not a priority? |

Context and the definition of reach: see the [README, II.1](https://github.com/theonlypgk/data2010-steam-chinese#1-business-understanding).

## 2.​ Data collection

- **Data:** Steam Games Dataset by FronkonGames (Hugging Face `FronkonGames/steam-games-dataset`), one row per game.
- **Two snapshots:** A = 2026-06-20, C = 2024-01-11, saved in `data/raw/` → same input on every run.
- **DATA ISSUE:** header of A merges two names (`DiscountDLC count`) → later columns shifted → repaired before reading.

### 2.1 The two snapshots

In [ ]:
# 2.1 · The two snapshots
# Reason: the dataset keeps changing -> the two files used are saved in data/raw
# Source: Hugging Face FronkonGames/steam-games-dataset, file games.csv
#   A = commit 52ba62abfa707a2fa31f4d147c2777a9d9fbd2da; C = commit aefdeb1ff1a739b92ac5338682cc2610a0328505

FILE_A = RAW + "games_a.csv"
FILE_C = RAW + "games_c.csv"
SNAPSHOT_DATE_A = "2026-06-20"                # day snapshot A was taken
SNAPSHOT_DATE_C = "2024-01-11"                # day snapshot C was taken

### 2.2 Read the two snapshots

In [ ]:
# 2.2 · Read the two snapshots
# DATA ISSUE: header of A merges two names ("DiscountDLC count") -> split it back, else columns shift
# Rule: read everything as text (dtype=str); fix types in 3.1.1

raw_c = pd.read_csv(FILE_C, dtype=str)

names_a = pd.read_csv(FILE_A, nrows=0).columns.tolist()     # header only
i = names_a.index("DiscountDLC count")
names_a[i:i + 1] = ["Discount", "DLC count"]                 # 1 name -> 2 names
raw_a = pd.read_csv(FILE_A, names=names_a, skiprows=1, dtype=str)

### 2.3 Keep the 12 columns the analysis uses

In [ ]:
# 2.3 · Keep the 12 columns the analysis uses, with English names
# Reason: 28 of the 40 columns do not serve the questions

# Columns kept -> used for:
#   AppID                 -> id; join C to A
#   Name                  -> check results by eye
#   Release date          -> year band; ">= 1 year old" filter
#   Price, Discount       -> price tier (list price)
#   Supported languages   -> treatment; other languages; Chinese-first rule 1; languages added (3.3)
#   Full audio languages  -> voice-over signal, only for the stricter filter of 5.1b
#   Positive, Negative    -> reach (number of reviews)
#   Developers            -> Chinese-first rule 2
#   Genres                -> Early Access flag; the list of Steam genres (3.1.6)
#   Tags                  -> genre group
# Not kept because:
#   Estimated owners                         -> ranges too wide
#   scores, playtime, CCU, achievements      -> measure quality/engagement, not reach
#   Publishers                               -> often regional partners, not the studio
#   Categories, age, DLC, platforms, notes   -> not part of the design
#   descriptions, links, media               -> no numbers
RAW_TO_CLEAN = {
    "AppID": "app_id",
    "Name": "name",
    "Release date": "release_date",
    "Price": "price",
    "Supported languages": "supported_languages",
    "Full audio languages": "full_audio_languages",
    "Positive": "positive",
    "Negative": "negative",
    "Developers": "developers",
    "Genres": "genres",
    "Tags": "tags",
}

raw_a = raw_a[list(RAW_TO_CLEAN) + ["Discount"]].rename(columns=RAW_TO_CLEAN | {"Discount": "discount_pct"})
raw_c = raw_c[list(RAW_TO_CLEAN)].rename(columns=RAW_TO_CLEAN)
raw_c["discount_pct"] = "0"                    # C has no Discount column
print(f"snapshot A: {len(raw_a):,} games x {raw_a.shape[1]} columns")
print(f"snapshot C: {len(raw_c):,} games x {raw_c.shape[1]} columns")
raw_a.head(3)

## 3.​ Data processing

| Part | What | Output |
|---|---|---|
| 3.1 Clean data | fix data issues, add analysis columns, filter | `pcd_games_clean` |
| 3.2 SQ1 groups | similar games, with vs without Chinese | `pcd_sq1_groups` |
| 3.3 SQ2 groups | similar games, added Chinese vs did not | `pcd_sq2_groups` |
| 3.4 Save the tables | write the output files | |

- **Rule:** snapshots A and C are cleaned the same way.

### 3.1 Clean data

- Cleans snapshots A (2026-06, for SQ1) and C (2024-01, for SQ2) the same way.
- Each step = a table of problems (a, b, c, ...), then its code cell; in the code, `# 3.1.1e · ...` stands above the lines that solve problem e.

#### Fix and add columns (3.1.1-3.1.8)

##### 3.1.1 · Fix data types and wrong values

| # | Problem | Solution | Output |
|---|---|---|---|
| a | id is text; some ids are not numbers | turn into a number; not a number → missing | `"570"` → `570`; `"abc"` → missing |
| b | missing id | drop the row | `app_id` missing → row dropped |
| c | repeated id (same game twice) | keep the first row | 2 rows with one id → 1 row |
| d | review counts are text; some are missing | turn into numbers; missing → 0 | `"1000"` → `1000`; `""` → `0` |
| e | price is text | turn into a number (the price on the snapshot day) | `"7.49"` → `7.49` |
| f | missing discount; a 100% discount divides by zero | missing → 0; cap at 99% | `100` → `99` |
| g | **DATA ISSUE:** snapshot A stores the sale price | list price = `price / (1 - discount / 100)` | Hades `7.49` at `-70%` → `24.99` |
| h | dates in several formats, some unreadable | read every format; unreadable → missing | `"Aug 1, 2023"` → `2023-08-01`; `"Coming soon"` → `NaT` |
| i | no release year and no game age | add `release_year` and `days_since_release` | `release_year = 2023`, `days_since_release` = days from release to the snapshot |

In [ ]:
# 3.1.1 · Fix data types and wrong values (problems a-i above)

def fix_types(trim, snapshot_date):
    """Give one snapshot real types, one row per game, list prices and dates.

    Args:
        trim: one snapshot from 2.3 (every value is text).
        snapshot_date: day the snapshot was taken, "YYYY-MM-DD".

    Returns:
        A new DataFrame, one row per app_id:
        app_id, positive, negative -> int64; price -> list price (float);
        release_date -> datetime (NaT if unreadable); new columns
        price_at_snapshot, release_year, days_since_release.
    """
    df = trim.copy()

    # 3.1.1a · id: text -> number; not a number -> missing
    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce")

    # 3.1.1b · missing id -> drop the row
    df = df.dropna(subset=["app_id"])

    # 3.1.1c · repeated id -> keep the first row
    df = df.drop_duplicates("app_id")
    df["app_id"] = df["app_id"].astype("int64")

    # 3.1.1d · review counts: text -> number; missing -> 0
    for col in ("positive", "negative"):
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")

    # 3.1.1e · price: text -> number (the price on the snapshot day)
    df["price_at_snapshot"] = pd.to_numeric(df["price"], errors="coerce")

    # 3.1.1f · discount: missing -> 0; capped at 99% because: 100% -> divide by zero
    discount = df["discount_pct"] if "discount_pct" in df else pd.Series(0, index=df.index)
    df["discount_pct"] = pd.to_numeric(discount, errors="coerce").fillna(0).clip(0, 99)

    # 3.1.1g · sale price -> list price
    df["price"] = (df["price_at_snapshot"] / (1 - df["discount_pct"] / 100)).round(2)

    # 3.1.1h · "Aug 1, 2023" or "Aug 2023" -> date; unreadable -> NaT (missing)
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce", format="mixed")

    # 3.1.1i · release year, and days from release to the snapshot
    df["release_year"] = df["release_date"].dt.year.astype("Int64")
    df["days_since_release"] = (pd.Timestamp(snapshot_date) - df["release_date"]).dt.days
    return df

##### 3.1.2 · Turn the list cells into clean lists

| # | Problem | Solution | Output |
|---|---|---|---|
| a | empty cell | empty list | `NaN` → `[]` |
| b | **DATA ISSUE:** store-page junk in the cells (e.g. BBCode, HTML tags) | remove it first | `"English[b]*[/b]"` → `"English"` |
| c | list cells are text, written two ways (e.g. `"['A', 'B']"`, `"A,B"`) | split into pieces | `"['English', 'French']"` → `"'English'"`, `" 'French'"` |
| d | pieces keep spaces and quotes | strip them | `" 'French'"` → `"French"` |
| e | empty pieces and repeated items | drop them, keep the order (tags are sorted by votes) | `["English", "English"]` → `["English"]` |

In [ ]:
# 3.1.2 · Turn the list cells into clean lists (problems a-e above)

# store-page junk (used in b): BBCode tags, HTML tags/entities, "*", line breaks, "#lang_..." codes
_TAG_JUNK = re.compile(r"\[/?[a-zA-Z]+\]|<[^>]+>|&amp;|&lt;|&gt;|\*|\r|\n|#lang_\w+")


def parse_list(value):
    """Turn one list-like text cell into a clean list.

    Args:
        value: a cell such as "['English', 'French']", "A,B", "English[b]*[/b]" or NaN.

    Returns:
        List of unique strings in the original order, e.g. ["English", "French"];
        [] for a missing cell.
    """

    # 3.1.2a · empty cell -> []
    if pd.isna(value):
        return []

    # 3.1.2b · remove the junk first
    text = _TAG_JUNK.sub("", str(value)).strip()

    # 3.1.2c · "['A', 'B']" or "A,B" -> pieces
    items = text.strip("[]").split(",")
    out = []
    for item in items:
        # 3.1.2d · drop spaces and quotes
        item = item.strip().strip("'\"").strip()

        # 3.1.2e · drop empties and repeats, keep the order
        if item and item != "nan" and item not in out:
            out.append(item)
    return out

##### 3.1.3 · Add the review count and language columns

| # | Problem | Solution | Output |
|---|---|---|---|
| a | **DATA ISSUE:** a language was renamed between C and A | use the new name in both snapshots (e.g. "Portuguese" → "Portuguese - Portugal") | `"Portuguese"` → `"Portuguese - Portugal"` |
| b | Steam publishes no sales | reach = positive + negative reviews | `900 + 100` → `total_reviews = 1000` |
| c | no Simplified Chinese column | look for it in the language list | `[..., "Simplified Chinese"]` → `has_zh_simplified = True` |
| d | no Traditional Chinese / English columns (needed in 3.1.4, 3.1.5) | look for them in the language list | `has_zh_traditional`, `has_english` (True/False) |
| e | bigger games ship in more languages | count the other languages (not Chinese, not English) | `["English", "French", "German", "Simplified Chinese"]` → `n_other_langs = 2` |
| f | 3.3 must see which languages a game added | keep all languages in one text column | `all_langs = "English\|French\|German\|Simplified Chinese"` |

In [ ]:
# 3.1.3 · Add the review count and language columns (problems a-f above)

# old name (C) -> new name (A), used in a
RENAMED_LANGS = {"Portuguese": "Portuguese - Portugal"}
ZH_SIMPLIFIED = "Simplified Chinese"
ZH_TRADITIONAL = "Traditional Chinese"
ENGLISH = "English"


def add_review_and_language_columns(df, langs):
    """Add the review count (reach) and the language columns.

    Args:
        df: snapshot after fix_types.
        langs: Series of clean language lists (parse_list of supported_languages).

    Returns:
        The same DataFrame with new columns:
        total_reviews (int), has_zh_simplified, has_zh_traditional, has_english (bool),
        n_other_langs (int), all_langs (str, e.g. "English|French").
    """

    # 3.1.3a · renamed language -> new name; no repeats
    langs = langs.map(lambda x: list(dict.fromkeys(RENAMED_LANGS.get(l, l) for l in x)))

    # 3.1.3b · reach = number of reviews
    df["total_reviews"] = df["positive"] + df["negative"]

    # 3.1.3c · Simplified Chinese (the treatment)
    df["has_zh_simplified"] = langs.map(lambda x: ZH_SIMPLIFIED in x)

    # 3.1.3d · Traditional Chinese and English
    df["has_zh_traditional"] = langs.map(lambda x: ZH_TRADITIONAL in x)
    df["has_english"] = langs.map(lambda x: ENGLISH in x)

    # 3.1.3e · other languages = all languages minus Chinese and English (set difference)
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["n_other_langs"] = langs.map(lambda x: len(set(x) - chinese - {ENGLISH}))

    # 3.1.3f · all languages in one text column: "English|French|..."
    df["all_langs"] = langs.map(lambda x: "|".join(sorted(x)))
    return df

##### 3.1.4 · Flag the Chinese-first games (removed in 3.1.9)

| # | Problem | Solution | Output |
|---|---|---|---|
| a | several developers in one cell | split on commas | `"A, B"` → `["A", "B"]` |
| b | a foreign game can list one Chinese co-developer | require every developer to match | `["Ubisoft", "腾讯"]` → `False` |
| c | Chinese studio with an English release | check that every developer name has Chinese characters | `"鬼谷工作室"` → `True` |
| d | Japanese names also use Chinese characters (kanji) | check for kana (only Japanese uses it) | `"株式会社ゲームフリーク"` → kana found |
| e | Chinese text without English text | rule 1 | `["Simplified Chinese"]` → `cn_text = True` |
| f | Chinese-character names that are not Japanese | rule 2 = c and not d | `"鬼谷工作室"` → `cn_dev_name = True` |
| g | for a Chinese studio, Chinese is not a translation | flag the game if rule 1 or rule 2 is true | `is_cn_game = True` |
| h | a stricter filter is tested later (5.1b) | compute 2 extra signals, not used to filter | `cn_dev_kana`, `cn_audio` (True/False) |

- **Not used:** hand-made lists of cities or companies (dataset too large to check row by row); voice-over languages (Japanese games are dubbed in Chinese too); publishers (often regional partners).
- **Missed:** Chinese studios with an English name and an English release (Black Myth: Wukong).

In [ ]:
# 3.1.4 · Flag the Chinese-first games (problems a-h above)

# a Chinese character (also used in Japanese kanji)
_HAN = re.compile(r"[一-鿿]")
# hiragana or katakana: only Japanese uses them
_KANA = re.compile(r"[぀-ヿ]")


def split_developers(text):
    """Split a developer cell into developer names.

    Args:
        text: developer cell, e.g. "A, B".

    Returns:
        List of names, e.g. ["A", "B"].
    """

    # 3.1.4a · split on commas
    return [p.strip() for p in str(text).split(",") if p.strip()]


def all_match(developers, pattern):
    """Check that every developer name matches a pattern.

    Args:
        developers: list of names from split_developers.
        pattern: compiled regex, e.g. _HAN (Chinese characters).

    Returns:
        True if the list is not empty and every name matches; otherwise False.
    """

    # 3.1.4b · every name, not any
    return bool(developers) and all(pattern.search(d) for d in developers)


def add_chinese_game_flags(df, audio):
    """Flag the Chinese-first games.

    Args:
        df: snapshot after add_review_and_language_columns.
        audio: Series of clean voice-over language lists.

    Returns:
        The same DataFrame with new columns:
        cn_text, cn_dev_name (bool, the 2 rules), cn_dev_kana, cn_audio (bool, not used to filter),
        is_cn_game (bool, cn_text or cn_dev_name).
    """
    developers = df["developers"].fillna("").astype(str)

    # 3.1.4c · every developer name has Chinese characters (uses a and b)
    han_name = developers.map(split_developers).map(lambda d: all_match(d, _HAN))

    # 3.1.4d · any kana in the developer cell -> Japanese
    kana_name = developers.str.contains(_KANA)

    # 3.1.4e · rule 1: Chinese text without English text
    df["cn_text"] = (df["has_zh_simplified"] | df["has_zh_traditional"]) & ~df["has_english"]

    # 3.1.4f · rule 2: Chinese-character names, no kana
    df["cn_dev_name"] = han_name & ~kana_name

    # 3.1.4g · Chinese-first game = rule 1 or rule 2
    df["is_cn_game"] = df["cn_text"] | df["cn_dev_name"]

    # 3.1.4h · 2 extra signals, only for the stricter filter of 5.1b
    chinese = {ZH_SIMPLIFIED, ZH_TRADITIONAL}
    df["cn_dev_kana"] = han_name & kana_name
    df["cn_audio"] = audio.map(lambda x: bool(set(x) & chinese) and ENGLISH not in x)
    return df

##### 3.1.5 · Flag Early Access and Traditional-Chinese-only games

| # | Problem | Solution | Output |
|---|---|---|---|
| a | Early Access = unfinished game | flag it (removed in 3.1.9) | `is_early_access = True` |
| b | Traditional Chinese only: neither "with" nor "without" Simplified Chinese | flag it (removed in 3.1.9) | `is_traditional_only = True` |

In [ ]:
# 3.1.5 · Flag Early Access and Traditional-Chinese-only games (problems a-b above)

def add_filter_flags(df, genres):
    """Flag the games removed later in 3.1.9.

    Args:
        df: snapshot after add_review_and_language_columns.
        genres: Series of clean genre lists.

    Returns:
        The same DataFrame with new columns is_early_access and is_traditional_only (bool).
    """

    # 3.1.5a
    df["is_early_access"] = genres.map(lambda x: "Early Access" in x)

    # 3.1.5b
    df["is_traditional_only"] = df["has_zh_traditional"] & ~df["has_zh_simplified"]
    return df

##### 3.1.6 · Assign a genre to each game

| # | Problem | Solution | Output |
|---|---|---|---|
| a | a hand-made genre list cannot be checked against 125k games | use Steam's own genre labels: share of games with each label | 9 genres: Action, Adventure, Casual, Massively Multiplayer, Racing, RPG, Simulation, Sports, Strategy |
| b | rare labels cannot form groups (e.g. content warnings, software tools) | keep labels on >= 2% of games | "Violent", "Utilities" → not genres |
| c | some labels are not a type of gameplay | drop them (e.g. Indie, Early Access, Free to Play) | "Indie" → not a genre |
| d | **DATA ISSUE:** the same label written with different upper/lower case (e.g. "Free To Play", "Free to Play") | compare in lower case | both dropped |
| e | several genres and tags per game | take the first user tag that is a genre (tags are sorted by votes) | `["Indie", "Adventure", "Puzzle", "Action"]` → `"Adventure"` |
| f | no tag is a genre | missing (removed in 3.1.9) | `["Indie", "Pixel Graphics"]` → `None` |

In [ ]:
# 3.1.6 · Assign a genre to each game (problems a-f above)

# labels that are not a type of gameplay (used in c), in lower case
NOT_GAMEPLAY = {"indie", "early access", "free to play"}
# smallest share of games a genre must have (used in b)
MIN_GENRE_SHARE = 0.02


def steam_genres(genre_lists, min_share=MIN_GENRE_SHARE):
    """Build the genre list from Steam's own genre labels.

    Args:
        genre_lists: Series of clean genre lists (parse_list of the genres column).
        min_share: smallest share of games a label must appear on.

    Returns:
        Sorted list of genre names, e.g. ["Action", "Adventure", ...]: the labels on
        >= min_share of the games, without the labels in NOT_GAMEPLAY (any upper/lower case).
    """

    # 3.1.6a · share of games with each Steam genre label
    share = genre_lists.explode().value_counts() / len(genre_lists)
    return sorted(
        g for g, s in share.items()
        # 3.1.6b · not rare
        if s >= min_share
        # 3.1.6c, d · not a type of gameplay; compared in lower case
        and g.lower() not in NOT_GAMEPLAY
    )


# the genre list, from snapshot A; used for A and C
GENRES = steam_genres(raw_a["genres"].map(parse_list))
print(f"{len(GENRES)} genres:", ", ".join(GENRES))


def first_genre(tags):
    """Give a game one genre group.

    Args:
        tags: clean list of user tags, most voted first.

    Returns:
        The first tag that is in GENRES, e.g. "RPG"; None if no tag is a genre.
    """

    # 3.1.6e · first tag that is a genre
    for tag in tags:
        if tag in GENRES:
            return tag

    # 3.1.6f · no tag is a genre
    return None

##### 3.1.7 · Encoding scheme

| # | Problem | Solution | Output |
|---|---|---|---|
| a | text labels sort wrongly (e.g. "10to20" before "5to10") | save each bin as its rank 1, 2, 3, ... | `"5to10"` → `3`, `"10to20"` → `4` |
| b | too many prices to form groups | 5 price tiers | `24.99` → `price_tier = 5` |
| c | too many release years | 4 year bands | `2019` → `year_band = 2` |
| d | too many language counts | 3 bins | `7` → `n_other_langs_bin = 3` |

| Code | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| `price_tier` | free | < \$5 | \$5-10 | \$10-20 | > \$20 |
| `year_band` | <=2017 | 2018-2019 | 2020-2021 | 2022+ | |
| `n_other_langs_bin` | 0 | 1-4 | 5+ | | |

- Genre is not encoded because: genres have no order.

In [ ]:
# 3.1.7 · Encoding scheme (problems a-d above)

# bins and their labels; code = position in the list, 1, 2, 3, ...
# price tiers, USD list price (codes 1-5)
PRICE_BINS = [-0.001, 0.0, 4.999, 9.999, 19.999, float("inf")]
PRICE_LABELS = ["free", "lt5", "5to10", "10to20", "gt20"]
# year bands (codes 1-4)
YEAR_BAND_BINS = [0, 2017, 2019, 2021, 2100]
YEAR_BAND_LABELS = ["<=2017", "2018-2019", "2020-2021", "2022+"]
# number of other languages (codes 1-3)
OTHER_LANG_BINS = [-1, 0, 4, 10_000]
OTHER_LANG_LABELS = ["0", "1-4", "5+"]


def to_code(values, bins):
    """Put numbers into bins and return the bin number.

    Args:
        values: Series of numbers, e.g. prices.
        bins: bin edges, e.g. PRICE_BINS.

    Returns:
        Series of Int64 codes 1, 2, 3, ... (missing stays missing).
    """

    # 3.1.7a · labels=False gives the bin position 0, 1, 2, ...; +1 makes it 1, 2, 3, ...
    return (pd.cut(values.astype("float"), bins, labels=False) + 1).astype("Int64")


def add_codes(df):
    """Add the binned columns as codes.

    Args:
        df: snapshot with price, release_year and n_other_langs.

    Returns:
        The same DataFrame with new columns price_tier (1-5), year_band (1-4),
        n_other_langs_bin (1-3), all Int64.
    """

    # 3.1.7b · price tier
    df["price_tier"] = to_code(df["price"], PRICE_BINS)

    # 3.1.7c · year band
    df["year_band"] = to_code(df["release_year"], YEAR_BAND_BINS)

    # 3.1.7d · other-languages bin
    df["n_other_langs_bin"] = to_code(df["n_other_langs"], OTHER_LANG_BINS)
    return df

##### 3.1.8 · Apply the cleaning to both snapshots

| # | Problem | Solution | Output |
|---|---|---|---|
| a | A and C must be cleaned the same way | run steps 3.1.1-3.1.7 on both | `feat_a` 125,855 games, `feat_c` 85,103 games (no game removed yet) |

In [ ]:
# 3.1.8 · Apply the cleaning to both snapshots (problem a above)

def add_features(trim, snapshot_date):
    """Run steps 3.1.1-3.1.7 on one snapshot.

    Args:
        trim: one snapshot from 2.3.
        snapshot_date: day the snapshot was taken, "YYYY-MM-DD".

    Returns:
        DataFrame, one row per game (no game removed yet), with every column
        added in 3.1.1-3.1.7.
    """

    # 3.1.1
    df = fix_types(trim, snapshot_date)

    # 3.1.2
    langs = df["supported_languages"].map(parse_list)
    audio = df["full_audio_languages"].map(parse_list)
    tags = df["tags"].map(parse_list)
    genres = df["genres"].map(parse_list)

    # 3.1.3
    df = add_review_and_language_columns(df, langs)

    # 3.1.4
    df = add_chinese_game_flags(df, audio)

    # 3.1.5
    df = add_filter_flags(df, genres)

    # 3.1.6
    df["genre_group"] = tags.map(first_genre)

    # 3.1.7
    return add_codes(df)


# 3.1.8a · the same function for both snapshots
feat_a = add_features(raw_a, SNAPSHOT_DATE_A)
feat_c = add_features(raw_c, SNAPSHOT_DATE_C)
print(f"A: {len(feat_a):,} games, C: {len(feat_c):,} games")

#### Filter the rows (3.1.9)

##### 3.1.9 · Filter out unusable games in A and C

| # | Problem | Solution | Output |
|---|---|---|---|
| a | missing value inside a filter | counts as failing | missing date → removed |
| b | unknown how many games each filter removes | apply the filters in order, count after each | funnel table below |
| c | Chinese-first games | drop | `is_cn_game = True` → removed |
| d | Early Access games | drop | `is_early_access = True` → removed |
| e | below 10 reviews, reach is noise; a higher threshold would keep only successful games | keep >= 10 reviews | 3 reviews → removed |
| f | new games had no time to collect reviews | keep games released >= 1 year before the snapshot | released 200 days before → removed |
| g | no genre, price or year → no group | drop | `genre_group = None` → removed |
| h | Traditional-Chinese-only games | drop (A only) | `is_traditional_only = True` → removed |
| i | C (for SQ2) must be filtered like A | filters c-g on C too | `kept_c`; one funnel table, a column for A and one for C |

- **Purpose of `apply_filters`:** shows how many rows each filter removes → explains why 125,855 games become the 47,603 analysed.

In [ ]:
# 3.1.9 · Filter out unusable games in A and C (problems a-i above)

# thresholds (used in e and f)
# below 10 reviews, reach is noise; not higher because: reviews are what we measure (reach)
MIN_REVIEWS = 10
# every game had a year to collect reviews
MIN_AGE_DAYS = 365


def apply_filters(df, steps, table):
    """Apply filters in order and count the games left after each one.

    Args:
        df: table to filter.
        steps: list of (filter name, Series True = keep), applied in order.
        table: name written in the funnel, e.g. "games_clean".

    Returns:
        (kept, funnel): kept = copy of the rows that pass every filter;
        funnel = DataFrame with columns table, step, games_left (one row per step).
    """
    rows = [{"table": table, "step": "start", "games_left": len(df)}]
    keep = pd.Series(True, index=df.index)
    for name, mask in steps:
        # 3.1.9a · missing value -> fails the filter
        keep &= mask.reindex(df.index).fillna(False)

        # 3.1.9b · count the games left after each filter
        rows.append({"table": table, "step": name, "games_left": int(keep.sum())})
    return df[keep].copy(), pd.DataFrame(rows)


def base_filters(df):
    """List the filters shared by both snapshots.

    Args:
        df: snapshot after add_features.

    Returns:
        List of (filter name, Series True = keep), in the order they are applied.
    """
    return [
        # 3.1.9c
        ("drop Chinese-first games (either of 2 signals)", ~df["is_cn_game"]),
        # 3.1.9d
        ("drop Early Access", ~df["is_early_access"]),
        # 3.1.9e
        (f"keep >= {MIN_REVIEWS} reviews", df["total_reviews"] >= MIN_REVIEWS),
        # 3.1.9f
        ("keep released >= 1 year before snapshot", df["days_since_release"] >= MIN_AGE_DAYS),
        # 3.1.9g
        ("keep games with a genre group, price and year",
         df["genre_group"].notna() & df["price"].notna() & df["release_year"].notna()),
    ]


# 3.1.9h · A: the shared filters + drop Traditional-Chinese-only games
steps_a = base_filters(feat_a) + [("drop Traditional-Chinese-only", ~feat_a["is_traditional_only"])]
kept_a, funnel_a = apply_filters(feat_a, steps_a, "snapshot A")

# 3.1.9i · C: the shared filters
kept_c, funnel_c = apply_filters(feat_c, base_filters(feat_c), "snapshot C")

# one funnel: games left after each step, A and C side by side (C has no step h)
funnel = funnel_a[["step", "games_left"]].merge(funnel_c[["step", "games_left"]], on="step", how="left", suffixes=("_A", "_C"))
funnel["games_left_C"] = funnel["games_left_C"].astype("Int64")
funnel

#### Build the table (3.1.10)

##### 3.1.10 · Build the cleaned table `pcd_games_clean` from snapshot A

- Only A, because: SQ1 looks at the current catalogue. Snapshot C (`kept_c`) is used in 3.3 for SQ2.

| # | Problem | Solution | Output |
|---|---|---|---|
| a | treatment is True/False | turn into 1/0 | `True` → `has_zh = 1` |
| b | review counts are very skewed | add the log10 of reviews | `1000` → `log_total_reviews = 3.0` |
| c | too many columns | keep the 12 the analysis uses | 12 columns, sorted by `app_id` |

In [ ]:
# 3.1.10 · Build the cleaned table pcd_games_clean from snapshot A (problems a-c above)

def make_games_clean(kept):
    """Turn the filtered games into pcd_games_clean.

    Args:
        kept: games of snapshot A left after 3.1.9 (kept_a).

    Returns:
        One row per game, 12 columns (app_id, name, genre_group, release_year, year_band, price,
        price_tier, n_other_langs, n_other_langs_bin, has_zh 0/1, total_reviews, log_total_reviews).
    """
    df = kept.copy()

    # 3.1.10a · True/False -> 1/0
    df["has_zh"] = df["has_zh_simplified"].astype(int)

    # 3.1.10b · rounded: same bytes every run
    df["log_total_reviews"] = np.log10(df["total_reviews"]).round(6)
    for col in ["price_tier", "year_band", "n_other_langs_bin"]:
        # no missing left after filters
        df[col] = df[col].astype(int)

    # 3.1.10c · the 12 columns
    cols = [
        "app_id", "name", "genre_group", "release_year", "year_band", "price", "price_tier",
        "n_other_langs", "n_other_langs_bin", "has_zh", "total_reviews", "log_total_reviews",
    ]
    return df[cols].sort_values("app_id").reset_index(drop=True)


games_clean = make_games_clean(kept_a)
print(f"games_clean: {len(games_clean):,} games, {games_clean['has_zh'].sum():,} with Simplified Chinese")
games_clean.head()

### 3.2 SQ1 groups → `pcd_sq1_groups`

- **Prepares for SQ1:** how much larger is the reach of games with Simplified Chinese, and in which genres?
- The two sides of each group are compared in section 5.1.

##### 3.2.1 · Assign each game to an SQ1 group

| # | Problem | Solution | Output |
|---|---|---|---|
| a | codes 1, 2, 3 are hard to read | write the label of each code in the group name | `1` → `"<=2017"` |
| b | bigger games are both translated and reviewed more | compare only inside groups of similar games: one group per game | `group_id` |
| c | audience and amount of text differ by genre | genre in the group | `"Action"` |
| d | older games had more time to collect reviews | year band in the group | `"<=2017"` |
| e | no budget or studio size in the data | price tier in the group | `"lt5"` |
| f | bigger games ship in more languages | other-languages bin in the group | `"5+"` → `"Action \| <=2017 \| lt5 \| 5+"` |

In [ ]:
# 3.2.1 · Assign each game to an SQ1 group (problems a-f above)

# 3.2.1a · code -> label, e.g. YEAR_BAND_LABEL[1] = "<=2017"
PRICE_LABEL = dict(enumerate(PRICE_LABELS, start=1))
YEAR_BAND_LABEL = dict(enumerate(YEAR_BAND_LABELS, start=1))
OTHER_LANGS_LABEL = dict(enumerate(OTHER_LANG_LABELS, start=1))


def add_sq1_group(games):
    """Give each game its SQ1 group.

    Args:
        games: pcd_games_clean (one row per game, with the codes of 3.1.7).

    Returns:
        A copy with a new column group_id, e.g. "Action | <=2017 | lt5 | 5+"
        (genre | year band | price tier | other languages).
    """

    # 3.2.1b · one group per game
    group_id = (
        # 3.2.1c · genre
        games["genre_group"]
        # 3.2.1d · year band
        + " | " + games["year_band"].map(YEAR_BAND_LABEL)
        # 3.2.1e · price tier
        + " | " + games["price_tier"].map(PRICE_LABEL)
        # 3.2.1f · other languages
        + " | " + games["n_other_langs_bin"].map(OTHER_LANGS_LABEL)
    )

    # copy: games_clean is not changed
    return games.assign(group_id=group_id)


games_sq1 = add_sq1_group(games_clean)
print(f"{games_sq1['group_id'].nunique()} groups")
games_sq1[["app_id", "name", "has_zh", "total_reviews", "group_id"]].head()

##### 3.2.2 · Count games and median reviews per SQ1 group

| # | Problem | Solution | Output |
|---|---|---|---|
| a | size of each side unknown | count games with and without Chinese | `n_zh`, `n_no_zh` |
| b | review counts are very skewed | median, not mean | `median_reviews_zh`, `median_reviews_no_zh` |
| c | a side with no game | count 0, median missing | `n_zh = 0` |
| d | with few games, one game moves the median | eligible = >= 10 games on each side | `eligible = True / False` |

In [ ]:
# 3.2.2 · Count games and median reviews per SQ1 group (problems a-d above)

# games needed on each side (used in d; also in 3.3.4)
MIN_GAMES_PER_SIDE = 10


def build_sq1_groups(games):
    """Count games and take median reviews on each side of each SQ1 group.

    Args:
        games: games with group_id (output of add_sq1_group).

    Returns:
        One row per group: group_id, genre_group, year_band, price_tier, n_other_langs_bin,
        n_zh, n_no_zh (int), median_reviews_zh, median_reviews_no_zh (float), eligible (bool);
        sorted by genre, then the codes.
    """
    keys = ["group_id", "genre_group", "year_band", "price_tier", "n_other_langs_bin"]
    wide = (
        games.groupby(keys + ["has_zh"])["total_reviews"]
        # 3.2.2a, b · count and median per group and side
        .agg(n="size", median="median")
        # sides become columns: (stat, 0/1)
        .unstack("has_zh")
    )
    out = pd.DataFrame({
        "n_zh": wide[("n", 1)],
        "n_no_zh": wide[("n", 0)],
        "median_reviews_zh": wide[("median", 1)],
        "median_reviews_no_zh": wide[("median", 0)],
    }).reset_index()

    # 3.2.2c · empty side -> 0
    out[["n_zh", "n_no_zh"]] = out[["n_zh", "n_no_zh"]].fillna(0).astype(int)

    # 3.2.2d
    out["eligible"] = (out["n_zh"] >= MIN_GAMES_PER_SIDE) & (out["n_no_zh"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)


sq1_groups = build_sq1_groups(games_sq1)
print(f"sq1_groups: {len(sq1_groups)} groups, {sq1_groups['eligible'].sum()} eligible")
sq1_groups.head()

### 3.3 SQ2 groups → `pcd_sq2_groups`

- **Prepares for SQ2:** does adding Simplified Chinese after launch still increase reach, and in which genres?
- Every game gains reviews over time anyway → compare games that added Chinese (treated) with similar games that changed nothing (control).
- Join game by game first, then group, because: growth is the change of **one** game between C and A.
- The two arms are compared in section 5.2.

##### 3.3.1 · Keep C games without Chinese that are also in A

| # | Problem | Solution | Output |
|---|---|---|---|
| a | C must be cleaned like A | start from `kept_c` (3.1.9) | 33,441 games |
| b | SQ2 is about games without Chinese before | keep games with no Chinese in C | |
| c | the game must exist in both snapshots | keep games also in A | `games_c`, funnel |

In [ ]:
# 3.3.1 · Keep C games without Chinese that are also in A (problems a-c above)

# 3.3.1a · start from snapshot C after 3.1.9
games_c = kept_c

steps_panel = [
    # 3.3.1b
    ("keep games with no Chinese (Simplified or Traditional) in C",
     ~(games_c["has_zh_simplified"] | games_c["has_zh_traditional"])),
    # 3.3.1c
    ("keep games also present in A", games_c["app_id"].isin(feat_a["app_id"])),
]
games_c, funnel_panel = apply_filters(games_c, steps_panel, "sq2_panel")
funnel_panel

##### 3.3.2 · Join each C game with the same game in A

| # | Problem | Solution | Output |
|---|---|---|---|
| a | need the state before adding Chinese | take genre, year, price and reviews from C | `reviews_c` |
| b | need the reviews of the same game later | join A by `app_id` | `reviews_a` |
| c | 0 reviews in A = delisted, -100% growth is not real | drop | |

In [ ]:
# 3.3.2 · Join each C game with the same game in A (problems a-c above)

games_c = games_c.set_index("app_id")

# 3.3.2a · characteristics and reviews before: from C
panel = games_c[["name", "genre_group", "release_year", "year_band", "price_tier"]].copy()
panel["price_c"] = games_c["price"]
panel["reviews_c"] = games_c["total_reviews"]

# 3.3.2b · the same games, same order, from A
games_a = feat_a.set_index("app_id").loc[games_c.index]
panel["reviews_a"] = games_a["total_reviews"]

# 3.3.2c · drop delisted games
panel = panel[panel["reviews_a"] > 0].copy()
games_a, games_c = games_a.loc[panel.index], games_c.loc[panel.index]
print(f"panel: {len(panel):,} games followed from C to A")

##### 3.3.3 · Compute growth and label treated / control / excluded

| # | Problem | Solution | Output |
|---|---|---|---|
| a | game added Simplified Chinese | treated | `English` → `English, Simplified Chinese` |
| b | game changed no language | control | `English, French` → `English, French` |
| c | game added another language (e.g. German, or only Traditional Chinese) | excluded | `English` → `English, German` |
| d | games start from very different sizes | growth = (reviews in A - reviews in C) / reviews in C | 100 → 150 reviews: `growth = 0.5` |

- The label is saved in the column `arm` (arm = side of the comparison).
- Not used: a hand-made list of "major" languages because: it cannot be justified from the data; "unchanged" needs no list.

In [ ]:
# 3.3.3 · Compute growth and label treated / control / excluded (problems a-d above)

def label_arm(a, c):
    """Label each game by what it added between snapshot C and snapshot A.

    Args:
        a: the games in snapshot A (rows in the same order as c).
        c: the same games in snapshot C, with no Chinese in C.

    Returns:
        Array of "treated", "control" or "excluded", one per game.
    """

    # languages in A but not in C
    added = [
        set(la.split("|")) - set(lc.split("|")) - {""}
        for la, lc in zip(a["all_langs"], c["all_langs"])
    ]

    # 3.3.3a · treated: added Simplified Chinese
    treated = a["has_zh_simplified"].to_numpy()

    # 3.3.3b · control: no language added
    unchanged = np.array([len(n) == 0 for n in added])

    # 3.3.3c · excluded: everything else (np.select: first True condition gives the label)
    return np.select([treated, unchanged], ["treated", "control"], default="excluded")


# arm of each game (rules a-c)
panel["arm"] = label_arm(games_a, games_c)

# 3.3.3d · growth of each game
panel["growth"] = ((panel["reviews_a"] - panel["reviews_c"]) / panel["reviews_c"]).round(6)

# no missing left after filters
for col in ["year_band", "price_tier"]:
    panel[col] = panel[col].astype(int)
sq2_panel = panel.reset_index()[["app_id", "name", "genre_group", "release_year", "year_band", "price_tier",
                                 "price_c", "reviews_c", "reviews_a", "growth", "arm"]]
sq2_panel = sq2_panel.sort_values("app_id").reset_index(drop=True)
print(sq2_panel["arm"].value_counts().to_string())
sq2_panel.head()

##### 3.3.4 · Count games and median growth per SQ2 group

| # | Problem | Solution | Output |
|---|---|---|---|
| a | growth differs by genre and release year | group = genre x year band | `"Action \| <=2017"` |
| b | excluded games are not a clean control | keep only treated and control | keep only treated and control | |
| c | growth is very skewed | median, not mean | `median_growth_treated`, `median_growth_control` |
| d | an arm with no game | count 0, median missing | `n_treated = 0` |
| e | with few games, one game moves the median | eligible = >= 10 games in each arm | `eligible = True / False` |

In [ ]:
# 3.3.4 · Count games and median growth per SQ2 group (problems a-e above)

def build_sq2_groups(panel):
    """Count games and take median growth of each arm in each SQ2 group.

    Args:
        panel: the SQ2 panel (sq2_panel, one row per game).

    Returns:
        One row per group: group_id_sq2, genre_group, year_band, n_treated, n_control (int),
        median_growth_treated, median_growth_control (float), eligible (bool);
        sorted by genre, then old -> new.
    """

    # 3.3.4a
    panel = panel.assign(group_id_sq2=panel["genre_group"] + " | " + panel["year_band"].map(YEAR_BAND_LABEL))
    keys = ["group_id_sq2", "genre_group", "year_band"]

    # 3.3.4b
    used = panel[panel["arm"].isin(["treated", "control"])]

    # 3.3.4c
    wide = used.groupby(keys + ["arm"])["growth"].agg(n="size", median="median").unstack("arm")
    out = pd.DataFrame({
        "n_treated": wide[("n", "treated")],
        "n_control": wide[("n", "control")],
        "median_growth_treated": wide[("median", "treated")].round(6),
        "median_growth_control": wide[("median", "control")].round(6),
    }).reset_index()

    # 3.3.4d
    out[["n_treated", "n_control"]] = out[["n_treated", "n_control"]].fillna(0).astype(int)

    # 3.3.4e
    out["eligible"] = (out["n_treated"] >= MIN_GAMES_PER_SIDE) & (out["n_control"] >= MIN_GAMES_PER_SIDE)
    return out.sort_values(keys[1:]).reset_index(drop=True)


sq2_groups = build_sq2_groups(sq2_panel)
print(f"sq2_groups: {len(sq2_groups)} groups, {sq2_groups['eligible'].sum()} eligible")
sq2_groups

### 3.4 Save the tables

- **Save:** write the three output tables to `data/processed/csv/`. (`data/processed/xlsx/` holds a copy to open and look at.)

In [ ]:
# 3.4.1 · Write the three tables as csv
# Rule: csv only because: every later step reads the csv
#   (data/processed/xlsx holds a copy of the same tables, only to open and look at)

tables = {
    # 3.1: cleaned data, one row per game
    "games_clean": games_clean,
    # 3.2: one row per SQ1 group
    "sq1_groups": sq1_groups,
    # 3.3: one row per SQ2 group
    "sq2_groups": sq2_groups,
}
for name, table in tables.items():
    table.to_csv(PROCESSED_CSV + PREFIX + name + ".csv", index=False)
    print(f"{PREFIX}{name}: {len(table):,} rows x {table.shape[1]} columns")

## 4.​ Data analysis / EDA

- Describes the three saved tables only; no comparison of the two sides (that is section 5).
- Reads only the saved tables; figures go to `report/figures/`.
- To run it alone: cell I.a, then 4a.

| Section | Reads |
| :--- | :--- |
| 4.1 Games | `pcd_games_clean` |
| 4.2 SQ1 groups | `pcd_sq1_groups` |
| 4.3 SQ2 groups | `pcd_sq2_groups` |

In [ ]:
# 4a · Load the saved tables
# Reason: Stage 4 reads only the saved tables -> can run on its own

import matplotlib.pyplot as plt

games_clean = pd.read_csv(PROCESSED_CSV + PREFIX + "games_clean.csv")
sq1_groups = pd.read_csv(PROCESSED_CSV + PREFIX + "sq1_groups.csv")
sq2_groups = pd.read_csv(PROCESSED_CSV + PREFIX + "sq2_groups.csv")
print(f"games_clean {len(games_clean):,} rows | sq1_groups {len(sq1_groups)} | sq2_groups {len(sq2_groups)}")

### 4.1 Games: `pcd_games_clean`

### 4.2 SQ1 groups: `pcd_sq1_groups`

### 4.3 SQ2 groups: `pcd_sq2_groups`

## 5.​ Data modelling

- Compares the two sides of each group and answers SQ1, SQ2, SQ3.
- Reads the saved tables loaded in 4a; figures go to `report/figures/`.
- Exception: 5.1b also needs section 3 in memory (Run All).

| Section | Reads |
| :--- | :--- |
| 5.1 SQ1 | `pcd_sq1_groups` |
| 5.2 SQ2 | `pcd_sq2_groups` |
| 5.3 SQ3 | 5.1 and 5.2, per genre |

### 5.1 SQ1: how much larger is the reach of games with Simplified Chinese, and in which genres?

#### Does the SQ1 gap depend on the Chinese-first filter? (5.1b)

- **Reason:** the filter misses some Chinese games (Black Myth: Wukong).
- **Rule:** build `pcd_games_clean` and the SQ1 groups again with no filter, a stricter filter, and without the top 1% games; compute the gap each time.
- Gap of a group = median with Chinese / median without - 1; overall gap = median over eligible groups.
- Runs on copies → nothing is changed or saved.

In [ ]:
# 5.1b · Compute the SQ1 gap again under other filters
# Reason: the filter misses some Chinese games -> test whether that changes the result
# Rule: no filter, a stricter filter, and without the top 1% games
# Needs: feat_a and the functions of section 3 (Run All)

TOP_SHARE = 0.01              # share of most-reviewed games dropped in the last row


def sq1_gap(groups):
    """Median over eligible groups of (median with Chinese / median without - 1)."""
    eligible = groups[groups["eligible"]]
    pct_diff = eligible["median_reviews_zh"] / eligible["median_reviews_no_zh"] - 1
    return round(float(pct_diff.median()), 3)


# Works on a copy -> feat_a and the saved tables are not changed.
def sq1_summary(label, is_cn_game, drop_top=0.0):
    """SQ1 under the given Chinese-first flag, optionally without the most-reviewed games."""
    feat = feat_a.assign(is_cn_game=is_cn_game)
    steps = base_filters(feat) + [("drop Traditional-Chinese-only", ~feat["is_traditional_only"])]
    games = make_games_clean(apply_filters(feat, steps, "games_clean")[0])
    if drop_top:
        games = games[games["total_reviews"] <= games["total_reviews"].quantile(1 - drop_top)]
    groups = build_sq1_groups(add_sq1_group(games))
    return {"filter": label, "games": len(games), "eligible_groups": int(groups["eligible"].sum()),
            "median_gap": sq1_gap(groups)}


chosen = feat_a["cn_text"] | feat_a["cn_dev_name"]
stricter = chosen | feat_a["cn_dev_kana"] | feat_a["cn_audio"]
sensitivity = pd.DataFrame([
    sq1_summary("no filter", chosen & False),
    sq1_summary("chosen: text or developer name", chosen),
    sq1_summary("stricter: + kana names and voice-over", stricter),
    sq1_summary("chosen, without the 1% most-reviewed games", chosen, drop_top=TOP_SHARE),
])
sensitivity

### 5.2 SQ2: does adding Simplified Chinese after launch still increase reach, and in which genres?

### 5.3 SQ3: for each genre, translate at launch, later, or not a priority?

## 6.​ Model deployment